In [1]:
from langchain_community.document_loaders import CSVLoader

C:\Users\GARVIT\AppData\Local\Temp\ipykernel_12176\3306274919.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader
c:\Users\GARVIT\OneDrive\Desktop\coding\AI\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


In [5]:
loader = CSVLoader("resolveAI_customer_support_dataset_v2.csv")
documents = loader.load()

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2214.24it/s]


In [7]:
vectorstore = FAISS.from_documents(
    documents,
    embeddings
)


In [8]:

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

In [11]:


print(docs[1].page_content)

ticket_id: T03298
customer_id: C10230
timestamp: 2026-08-03 22:35:00
product: E-commerce
plan: Standard
region: Bengaluru
customer_tenure_months: 2
customer_age_group: 35-44
ticket_channel: Email
complaint_text: My order was cancelled without my approval
complaint_category: Order Cancellation
sub_category: Order Cancellation
sentiment: Negative
sentiment_score: -0.34
urgency: Low
priority: P3
previous_ticket_count: 0
same_issue_count: 0
previous_resolution_time_hours: 8
current_resolution_time_hours: 10
agent_id: AG106
resolution_status: Resolved
resolution_text: Investigated order cancellation issue; payment-order reconciliation failure identified and corrective action applied.
customer_satisfaction: 3.2
refund_requested: True
refund_amount: 15358.12
service_downtime_minutes: 27
usage_drop_percent: 0.0
payment_failure_count: 2
competitor_mentioned: False
churn_risk: Low
churned: No
root_cause: Payment-order reconciliation failure
root_cause_confidence: 0.97
date: 2026-08-03
month: 202

In [12]:
vectorstore.save_local("faiss_index")

In [13]:
import re

ID_PATTERNS = {
    "ticket_id": r"\bT\d+\b",
    "customer_id": r"\bC\d+\b",
    "order_id": r"\bORD\d+\b",
    "payment_id": r"\bPAY\d+\b",
    "refund_id": r"\bREF\d+\b",
}


def extract_ids(text):
    found_ids = {}

    for id_type, pattern in ID_PATTERNS.items():
        matches = re.findall(pattern, text, re.IGNORECASE)

        if matches:
            found_ids[id_type] = [match.upper() for match in matches]

    return found_ids

In [14]:
query = "What happened with customer C10230 and ticket T03298?"

result = extract_ids(query)

print(result)

{'ticket_id': ['T03298'], 'customer_id': ['C10230']}


In [15]:
query = "What is the status of order ORD100291?"

print(extract_ids(query))

{'order_id': ['ORD100291']}


In [21]:
import pandas as pd

df = pd.read_csv("resolveAI_customer_support_dataset_v2.csv")

id_columns = [
    "ticket_id",
    "customer_id",
    "order_id",
    "payment_id",
    "refund_id"
]

for column in id_columns:
    df[column] = df[column].astype(str).str.upper().str.strip()

print(df.shape)

(10000, 47)


In [22]:
def get_user_info(query):
    # Step 1: Extract IDs from the query
    ids = extract_ids(query)

    # Step 2: Check if customer ID was found
    if "customer_id" not in ids:
        return "No customer ID found in the query."

    customer_id = ids["customer_id"][0]

    # Step 3: Find the customer in the dataset
    result = df[df["customer_id"] == customer_id]

    # Step 4: Check if customer exists
    if result.empty:
        return f"No customer found with ID {customer_id}"

    # Step 5: Return the customer's information
    return result.iloc[0]

In [23]:
query = "Tell me the information of customer C10230"

user_info = get_user_info(query)

print(user_info)

ticket_id                                                                    T01902
customer_id                                                                  C10230
timestamp                                                       2026-01-04 14:18:00
product                                                                    Internet
plan                                                                          Basic
region                                                                      Kolkata
customer_tenure_months                                                           63
customer_age_group                                                            45-54
ticket_channel                                                                Email
complaint_text                                              Router keeps restarting
complaint_category                                                           Router
sub_category                                                                

In [24]:
def get_customer_history(customer_id):
    customer_id = customer_id.upper().strip()

    history = df[df["customer_id"] == customer_id]

    if history.empty:
        return None

    return history.sort_values("timestamp")

In [25]:
customer_history = get_customer_history("C10230")

print(customer_history)

     ticket_id customer_id            timestamp      product        plan  \
123     T01902      C10230  2026-01-04 14:18:00     Internet       Basic   
2377    T03964      C10230  2026-03-16 08:41:00  Credit Card  Enterprise   
4189    T04696      C10230  2026-05-13 22:38:00  Credit Card     Premium   
4944    T05874      C10230  2026-06-07 01:03:00  Credit Card  Enterprise   
6669    T06639      C10230  2026-08-01 02:53:00     Internet    Standard   
6754    T03298      C10230  2026-08-03 22:35:00   E-commerce    Standard   
7363    T07537      C10230  2026-08-16 18:31:00   Mobile App       Basic   
7398    T04436      C10230  2026-08-17 13:07:00  Credit Card    Standard   
9086    T00771      C10230  2026-09-05 03:09:00   Mobile App  Enterprise   

         region  customer_tenure_months customer_age_group ticket_channel  \
123     Kolkata                      63              45-54          Email   
2377  Ahmedabad                      66              18-24   Social Media   
4189  Be

In [27]:
history_summary = customer_history[
    [
        "ticket_id",
        "timestamp",
        "complaint_text",
        "root_cause",
        "root_cause_confidence",
        "resolution_status",
        "resolution_text",
        "refund_requested",
        "refund_status",
        "customer_satisfaction"
    ]
]

print(history_summary.to_string(index=False))

ticket_id           timestamp                                  complaint_text                           root_cause  root_cause_confidence resolution_status                                                                                                       resolution_text  refund_requested refund_status  customer_satisfaction
   T01902 2026-01-04 14:18:00                         Router keeps restarting                     Hardware failure                   0.80          Resolved                                 Investigated router issue; hardware failure identified and corrective action applied.             False NOT_REQUESTED                    3.2
   T03964 2026-03-16 08:41:00           Card payment failed at several stores           Risk engine false positive                   0.97          Resolved                  Investigated transaction issue; risk engine false positive identified and corrective action applied.             False NOT_REQUESTED                    3.2
   T04696 202

In [28]:
from typing import TypedDict, Optional, List, Dict, Any


class SupportState(TypedDict, total=False):
    query: str

    customer_id: Optional[str]
    order_id: Optional[str]
    ticket_id: Optional[str]

    current_case: Dict[str, Any]
    customer_history: List[Dict[str, Any]]
    similar_cases: List[Dict[str, Any]]

    final_response: str

In [29]:
def identify_entities(state: SupportState):

    query = state["query"]

    ids = extract_ids(query)

    return {
        "customer_id": ids.get("customer_id", [None])[0],
        "order_id": ids.get("order_id", [None])[0],
        "ticket_id": ids.get("ticket_id", [None])[0],
    }

In [30]:
state = {
    "query": "Tell me about customer C10230 and ticket T03298"
}

result = identify_entities(state)

print(result)

{'customer_id': 'C10230', 'order_id': None, 'ticket_id': 'T03298'}


In [31]:
def retrieve_customer_history(state: SupportState):

    customer_id = state.get("customer_id")

    if not customer_id:
        return {
            "customer_history": []
        }

    history = get_customer_history(customer_id)

    if history is None:
        return {
            "customer_history": []
        }

    # Convert DataFrame rows into dictionaries
    history_records = history.to_dict(orient="records")

    return {
        "customer_history": history_records
    }

In [32]:
state = {
    "query": "Tell me about customer C10230 and ticket T03298",
    "customer_id": "C10230",
    "ticket_id": "T03298"
}

result = retrieve_customer_history(state)

print(len(result["customer_history"]))
print(result["customer_history"][0])

9
{'ticket_id': 'T01902', 'customer_id': 'C10230', 'timestamp': '2026-01-04 14:18:00', 'product': 'Internet', 'plan': 'Basic', 'region': 'Kolkata', 'customer_tenure_months': 63, 'customer_age_group': '45-54', 'ticket_channel': 'Email', 'complaint_text': 'Router keeps restarting', 'complaint_category': 'Router', 'sub_category': 'Router', 'sentiment': 'Negative', 'sentiment_score': -0.47, 'urgency': 'Low', 'priority': 'P3', 'previous_ticket_count': 2, 'same_issue_count': 1, 'previous_resolution_time_hours': 9, 'current_resolution_time_hours': 7, 'agent_id': 'AG113', 'resolution_status': 'Resolved', 'resolution_text': 'Investigated router issue; hardware failure identified and corrective action applied.', 'customer_satisfaction': 3.2, 'refund_requested': False, 'refund_amount': 0.0, 'service_downtime_minutes': 21, 'usage_drop_percent': 0.0, 'payment_failure_count': 0, 'competitor_mentioned': False, 'churn_risk': 'Low', 'churned': 'No', 'root_cause': 'Hardware failure', 'root_cause_confide

In [33]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


builder = StateGraph(SupportState)

builder.add_node("identify_entities", identify_entities)
builder.add_node("retrieve_customer_history", retrieve_customer_history)

builder.add_edge(START, "identify_entities")
builder.add_edge("identify_entities", "retrieve_customer_history")
builder.add_edge("retrieve_customer_history", END)


memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [34]:
config = {
    "configurable": {
        "thread_id": "customer_C10230"
    }
}

result = graph.invoke(
    {
        "query": "Tell me about customer C10230 and ticket T03298"
    },
    config
)

print(result["customer_id"])
print(result["ticket_id"])
print(len(result["customer_history"]))

C10230
T03298
9


In [35]:
def retrieve_similar_cases(state: SupportState):

    # Get the current ticket/customer information
    ticket_id = state.get("ticket_id")

    if ticket_id:
        current_row = df[df["ticket_id"] == ticket_id]

        if not current_row.empty:
            complaint = current_row.iloc[0]["complaint_text"]
        else:
            complaint = state["query"]
    else:
        complaint = state["query"]

    # Search FAISS for semantically similar cases
    docs = vectorstore.similarity_search(
        complaint,
        k=5
    )

    similar_cases = []

    for doc in docs:
        similar_cases.append({
            "content": doc.page_content,
            "metadata": doc.metadata
        })

    return {
        "similar_cases": similar_cases
    }

In [36]:
from langchain_core.documents import Document

documents_with_metadata = []

for _, row in df.iterrows():

    text = f"""
Ticket ID: {row['ticket_id']}

Customer ID: {row['customer_id']}

Complaint:
{row['complaint_text']}

Complaint Category:
{row['complaint_category']}

Sub Category:
{row['sub_category']}

Root Cause:
{row['root_cause']}

Root Cause Confidence:
{row['root_cause_confidence']}

Resolution Status:
{row['resolution_status']}

Resolution:
{row['resolution_text']}

Refund Requested:
{row['refund_requested']}

Refund Status:
{row['refund_status']}
"""

    documents_with_metadata.append(
        Document(
            page_content=text,
            metadata={
                "ticket_id": str(row["ticket_id"]),
                "customer_id": str(row["customer_id"]),
                "order_id": str(row["order_id"]),
                "complaint_category": str(row["complaint_category"]),
                "sub_category": str(row["sub_category"]),
                "resolution_status": str(row["resolution_status"])
            }
        )
    )

print(len(documents_with_metadata))

10000


In [37]:
vectorstore = FAISS.from_documents(
    documents_with_metadata,
    embeddings
)

In [38]:
vectorstore.save_local("faiss_index")

In [42]:
query = "Customer C10230 says my payment was deducted but my order was cancelled"

docs = vectorstore.similarity_search(
    query,
    k=5
)

for i, doc in enumerate(docs, 1):
    print(f"\n{'='*60}")
    print(f"SIMILAR CASE {i}")
    print(f"{'='*60}")

    print("Ticket ID    :", doc.metadata.get("ticket_id"))
    print("Customer ID  :", doc.metadata.get("customer_id"))
    print("Order ID     :", doc.metadata.get("order_id"))
    print("Category     :", doc.metadata.get("complaint_category"))
    print("Sub-category :", doc.metadata.get("sub_category"))
    print("Status       :", doc.metadata.get("resolution_status"))

    print("\nCASE CONTENT:")
    print(doc.page_content)


SIMILAR CASE 1
Ticket ID    : T09007
Customer ID  : C11724
Order ID     : ORD-100084
Category     : Order Cancellation
Sub-category : Order Cancellation
Status       : Resolved

CASE CONTENT:

Ticket ID: T09007

Customer ID: C11724

Complaint:
The order was cancelled after payment was completed

Complaint Category:
Order Cancellation

Sub Category:
Order Cancellation

Root Cause:
Payment-order reconciliation failure

Root Cause Confidence:
0.93

Resolution Status:
Resolved

Resolution:
Investigated order cancellation issue; payment-order reconciliation failure identified and corrective action applied.

Refund Requested:
True

Refund Status:
COMPLETED


SIMILAR CASE 2
Ticket ID    : T02733
Customer ID  : C11140
Order ID     : ORD-102877
Category     : Order Cancellation
Sub-category : Order Cancellation
Status       : Resolved

CASE CONTENT:

Ticket ID: T02733

Customer ID: C11140

Complaint:
The order was cancelled after payment was completed

Complaint Category:
Order Cancellation

S

In [44]:
query = "Customer C10230 says my payment was deducted but my order was cancelled"

docs = vectorstore.similarity_search(
    query,
    k=5,
    filter={"customer_id": "C10230"}
)

for i, doc in enumerate(docs, 1):
    print(f"\n{'='*60}")
    print(f"CASE {i}")
    print(f"{'='*60}")

    print("Ticket ID    :", doc.metadata.get("ticket_id"))
    print("Customer ID  :", doc.metadata.get("customer_id"))
    print("Order ID     :", doc.metadata.get("order_id"))
    print("Category     :", doc.metadata.get("complaint_category"))
    print("Sub-category :", doc.metadata.get("sub_category"))
    print("Status       :", doc.metadata.get("resolution_status"))

    print("\n", doc.page_content)

In [45]:
vectorstore = FAISS.from_documents(
    documents_with_metadata,
    embeddings
)

In [46]:
vectorstore.save_local("faiss_index")

In [47]:
print(documents_with_metadata[0].metadata)

{'ticket_id': 'T07990', 'customer_id': 'C11125', 'order_id': 'ORD-100000', 'complaint_category': 'Playback', 'sub_category': 'Playback', 'resolution_status': 'Resolved'}


In [48]:
query = "Customer C10230 says my payment was deducted but my order was cancelled"

docs = vectorstore.similarity_search(
    query,
    k=5,
    filter={"customer_id": "C10230"}
)

print("Retrieved cases:", len(docs))

for doc in docs:
    print(
        doc.metadata["ticket_id"],
        "|",
        doc.metadata["customer_id"]
    )

Retrieved cases: 0


In [49]:
query = "Customer C10230 says my payment was deducted but my order was cancelled"

docs = vectorstore.similarity_search(
    query,
    k=5,
    filter={"customer_id": "C10230"},
    fetch_k=10000
)

print("Retrieved cases:", len(docs))

for doc in docs:
    print(
        doc.metadata["ticket_id"],
        "|",
        doc.metadata["customer_id"]
    )

Retrieved cases: 5
T03298 | C10230
T07537 | C10230
T06639 | C10230
T03964 | C10230
T04696 | C10230


In [50]:
for i, doc in enumerate(docs, 1):
    print(f"\n{'=' * 70}")
    print(f"CASE {i}")
    print(f"{'=' * 70}")

    print("Ticket ID     :", doc.metadata.get("ticket_id"))
    print("Customer ID   :", doc.metadata.get("customer_id"))
    print("Order ID      :", doc.metadata.get("order_id"))
    print("Category      :", doc.metadata.get("complaint_category"))
    print("Sub-category  :", doc.metadata.get("sub_category"))
    print("Status        :", doc.metadata.get("resolution_status"))

    print("\nCASE CONTENT:")
    print(doc.page_content)


CASE 1
Ticket ID     : T03298
Customer ID   : C10230
Order ID      : ORD-106754
Category      : Order Cancellation
Sub-category  : Order Cancellation
Status        : Resolved

CASE CONTENT:

Ticket ID: T03298

Customer ID: C10230

Complaint:
My order was cancelled without my approval

Complaint Category:
Order Cancellation

Sub Category:
Order Cancellation

Root Cause:
Payment-order reconciliation failure

Root Cause Confidence:
0.97

Resolution Status:
Resolved

Resolution:
Investigated order cancellation issue; payment-order reconciliation failure identified and corrective action applied.

Refund Requested:
True

Refund Status:
COMPLETED


CASE 2
Ticket ID     : T07537
Customer ID   : C10230
Order ID      : ORD-107363
Category      : Payment
Sub-category  : Payment
Status        : Resolved

CASE CONTENT:

Ticket ID: T07537

Customer ID: C10230

Complaint:
Payment is stuck and has not been completed

Complaint Category:
Payment

Sub Category:
Payment

Root Cause:
Gateway latency

Roo

In [51]:
def retrieve_organizational_memory(query, k=5):

    docs = vectorstore.similarity_search(
        query,
        k=k,
        filter={"resolution_status": "Resolved"},
        fetch_k=10000
    )

    return docs

In [52]:
query = "My payment was deducted but my order was cancelled"

org_cases = retrieve_organizational_memory(query)

print("Retrieved organizational cases:", len(org_cases))

Retrieved organizational cases: 5


In [53]:
for i, doc in enumerate(org_cases, 1):

    print(f"\n{'=' * 70}")
    print(f"ORGANIZATIONAL CASE {i}")
    print(f"{'=' * 70}")

    print("Ticket ID    :", doc.metadata.get("ticket_id"))
    print("Customer ID  :", doc.metadata.get("customer_id"))
    print("Category     :", doc.metadata.get("complaint_category"))
    print("Status       :", doc.metadata.get("resolution_status"))

    print("\n", doc.page_content)


ORGANIZATIONAL CASE 1
Ticket ID    : T04044
Customer ID  : C10542
Category     : Order Cancellation
Status       : Resolved

 
Ticket ID: T04044

Customer ID: C10542

Complaint:
My order was cancelled without my approval

Complaint Category:
Order Cancellation

Sub Category:
Order Cancellation

Root Cause:
Payment-order reconciliation failure

Root Cause Confidence:
0.9

Resolution Status:
Resolved

Resolution:
Investigated order cancellation issue; payment-order reconciliation failure identified and corrective action applied.

Refund Requested:
True

Refund Status:
COMPLETED


ORGANIZATIONAL CASE 2
Ticket ID    : T04469
Customer ID  : C11331
Category     : Order Cancellation
Status       : Resolved

 
Ticket ID: T04469

Customer ID: C11331

Complaint:
My order was cancelled without my approval

Complaint Category:
Order Cancellation

Sub Category:
Order Cancellation

Root Cause:
Payment-order reconciliation failure

Root Cause Confidence:
0.84

Resolution Status:
Resolved

Resolution

In [54]:
from typing import TypedDict, Optional, List, Dict, Any

class SupportState(TypedDict, total=False):

    query: str

    customer_id: Optional[str]
    order_id: Optional[str]
    ticket_id: Optional[str]

    current_case: Dict[str, Any]

    customer_history: List[Dict[str, Any]]

    similar_cases: List[Dict[str, Any]]

    transaction_data: Dict[str, Any]

    final_response: str

In [55]:
def investigate_case(state: SupportState):

    customer_id = state.get("customer_id")
    ticket_id = state.get("ticket_id")
    order_id = state.get("order_id")
    query = state["query"]

    # -----------------------------
    # 1. Customer History
    # -----------------------------

    customer_history = []

    if customer_id:

        history = get_customer_history(customer_id)

        if history is not None:
            customer_history = history.to_dict(orient="records")


    # -----------------------------
    # 2. Current Case
    # -----------------------------

    current_case = {}

    if ticket_id:

        result = find_by_id("ticket_id", ticket_id)

        if result is not None:
            current_case = result.to_dict()

    elif order_id:

        result = find_by_id("order_id", order_id)

        if result is not None:
            current_case = result.to_dict()


    # -----------------------------
    # 3. Organizational Memory
    # -----------------------------

    similar_cases = retrieve_organizational_memory(
        query,
        k=5
    )

    similar_cases_data = []

    for doc in similar_cases:

        similar_cases_data.append({
            "ticket_id": doc.metadata.get("ticket_id"),
            "customer_id": doc.metadata.get("customer_id"),
            "complaint_category": doc.metadata.get("complaint_category"),
            "sub_category": doc.metadata.get("sub_category"),
            "resolution_status": doc.metadata.get("resolution_status"),
            "content": doc.page_content
        })


    # -----------------------------
    # 4. Transaction Data
    # -----------------------------

    transaction_data = {}

    if order_id:

        result = find_by_id("order_id", order_id)

        if result is not None:

            transaction_data = {
                "order_id": result["order_id"],
                "order_status": result["order_status"],
                "order_amount": result["order_amount"],
                "payment_id": result["payment_id"],
                "payment_status": result["payment_status"],
                "refund_id": result["refund_id"],
                "refund_status": result["refund_status"],
                "refund_amount": result["refund_amount_synthetic"]
            }


    # -----------------------------
    # 5. Return Investigation
    # -----------------------------

    return {
        "current_case": current_case,
        "customer_history": customer_history,
        "similar_cases": similar_cases_data,
        "transaction_data": transaction_data
    }

In [56]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

builder = StateGraph(SupportState)

builder.add_node("identify_entities", identify_entities)
builder.add_node("investigate_case", investigate_case)

builder.add_edge(START, "identify_entities")
builder.add_edge("identify_entities", "investigate_case")
builder.add_edge("investigate_case", END)

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [57]:
config = {
    "configurable": {
        "thread_id": "customer_C10230_test"
    }
}

result = graph.invoke(
    {
        "query": "Customer C10230 says my payment was deducted but my order was cancelled. Ticket T03298"
    },
    config
)

print("CUSTOMER ID:", result["customer_id"])
print("TICKET ID:", result["ticket_id"])

print("\nCURRENT CASE:")
print(result["current_case"])

print("\nCUSTOMER HISTORY:")
print("Number of previous records:", len(result["customer_history"]))

print("\nSIMILAR ORGANIZATIONAL CASES:")
print("Number of cases:", len(result["similar_cases"]))

print("\nTRANSACTION DATA:")
print(result["transaction_data"])

NameError: name 'find_by_id' is not defined

In [58]:
def find_by_id(id_type, id_value):

    if id_type not in df.columns:
        return None

    result = df[df[id_type] == id_value.upper().strip()]

    if result.empty:
        return None

    return result.iloc[0]

In [59]:
config = {
    "configurable": {
        "thread_id": "customer_C10230_test"
    }
}

result = graph.invoke(
    {
        "query": "Customer C10230 says my payment was deducted but my order was cancelled. Ticket T03298"
    },
    config
)

print("CUSTOMER ID:", result["customer_id"])
print("TICKET ID:", result["ticket_id"])

print("\nCURRENT CASE:")
print(result["current_case"])

print("\nCUSTOMER HISTORY:")
print("Number of previous records:", len(result["customer_history"]))

print("\nSIMILAR ORGANIZATIONAL CASES:")
print("Number of cases:", len(result["similar_cases"]))

print("\nTRANSACTION DATA:")
print(result["transaction_data"])

CUSTOMER ID: C10230
TICKET ID: T03298

CURRENT CASE:
{'ticket_id': 'T03298', 'customer_id': 'C10230', 'timestamp': '2026-08-03 22:35:00', 'product': 'E-commerce', 'plan': 'Standard', 'region': 'Bengaluru', 'customer_tenure_months': 2, 'customer_age_group': '35-44', 'ticket_channel': 'Email', 'complaint_text': 'My order was cancelled without my approval', 'complaint_category': 'Order Cancellation', 'sub_category': 'Order Cancellation', 'sentiment': 'Negative', 'sentiment_score': -0.34, 'urgency': 'Low', 'priority': 'P3', 'previous_ticket_count': 0, 'same_issue_count': 0, 'previous_resolution_time_hours': 8, 'current_resolution_time_hours': 10, 'agent_id': 'AG106', 'resolution_status': 'Resolved', 'resolution_text': 'Investigated order cancellation issue; payment-order reconciliation failure identified and corrective action applied.', 'customer_satisfaction': 3.2, 'refund_requested': True, 'refund_amount': 15358.12, 'service_downtime_minutes': 27, 'usage_drop_percent': 0.0, 'payment_fail

In [60]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


# --------------------------------
# 1. Structured output for reasoning
# --------------------------------

class CaseReasoning(BaseModel):

    issue_summary: str = Field(
        description="Short summary of the customer's issue"
    )

    likely_root_cause: str = Field(
        description="Most likely root cause based only on the available evidence"
    )

    evidence: List[str] = Field(
        description="Important evidence supporting the conclusion"
    )

    confidence: str = Field(
        description="Confidence level: Low, Medium, or High"
    )

    recommended_action: str = Field(
        description="Recommended next action based on the evidence"
    )


# --------------------------------
# 2. Create LLM
# --------------------------------

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

structured_llm = llm.with_structured_output(CaseReasoning)


# --------------------------------
# 3. Reasoning Prompt
# --------------------------------

reasoning_prompt = ChatPromptTemplate.from_messages([

    (
        "system",
        """
You are the reasoning engine of an autonomous customer support system.

Analyze the customer's issue using ONLY the evidence provided.

Do not invent customer, transaction, policy, or historical information.

Compare:
1. Current case
2. Customer history
3. Similar resolved organizational cases
4. Transaction information

Identify the most likely root cause.

If the evidence conflicts or is insufficient, state that clearly
and use a lower confidence level.

Do not perform any action yourself.
Only recommend the next appropriate action.
"""
    ),

    (
        "human",
        """
CUSTOMER QUERY:
{query}

CURRENT CASE:
{current_case}

CUSTOMER HISTORY:
{customer_history}

SIMILAR ORGANIZATIONAL CASES:
{similar_cases}

TRANSACTION DATA:
{transaction_data}
"""
    )
])


# --------------------------------
# 4. Reasoning Node
# --------------------------------

def reason_about_case(state: SupportState):

    chain = reasoning_prompt | structured_llm

    reasoning = chain.invoke({
        "query": state["query"],
        "current_case": state.get("current_case", {}),
        "customer_history": state.get("customer_history", []),
        "similar_cases": state.get("similar_cases", []),
        "transaction_data": state.get("transaction_data", {})
    })

    return {
        "case_reasoning": reasoning.model_dump()
    }

In [61]:
# Update the LangGraph with the reasoning node

builder = StateGraph(SupportState)

builder.add_node("identify_entities", identify_entities)
builder.add_node("investigate_case", investigate_case)
builder.add_node("reason_about_case", reason_about_case)

builder.add_edge(START, "identify_entities")
builder.add_edge("identify_entities", "investigate_case")
builder.add_edge("investigate_case", "reason_about_case")
builder.add_edge("reason_about_case", END)

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [62]:
config = {
    "configurable": {
        "thread_id": "customer_C10230_reasoning_test"
    }
}

result = graph.invoke(
    {
        "query": "Customer C10230 says my payment was deducted but my order was cancelled. Ticket T03298"
    },
    config
)

print("CUSTOMER ID:", result["customer_id"])
print("TICKET ID:", result["ticket_id"])

print("\n========== CASE REASONING ==========\n")

print("Issue Summary:")
print(result["case_reasoning"]["issue_summary"])

print("\nLikely Root Cause:")
print(result["case_reasoning"]["likely_root_cause"])

print("\nEvidence:")
for evidence in result["case_reasoning"]["evidence"]:
    print("-", evidence)

print("\nConfidence:")
print(result["case_reasoning"]["confidence"])

print("\nRecommended Action:")
print(result["case_reasoning"]["recommended_action"])

CUSTOMER ID: C10230
TICKET ID: T03298

========== CASE REASONING ==========

Issue Summary:


KeyError: 'case_reasoning'

In [63]:
from typing import TypedDict, Optional, List, Dict, Any

class SupportState(TypedDict, total=False):

    query: str

    customer_id: Optional[str]
    order_id: Optional[str]
    ticket_id: Optional[str]

    current_case: Dict[str, Any]

    customer_history: List[Dict[str, Any]]

    similar_cases: List[Dict[str, Any]]

    transaction_data: Dict[str, Any]

    case_reasoning: Dict[str, Any]

    final_response: str

In [64]:
builder = StateGraph(SupportState)

builder.add_node("identify_entities", identify_entities)
builder.add_node("investigate_case", investigate_case)
builder.add_node("reason_about_case", reason_about_case)

builder.add_edge(START, "identify_entities")
builder.add_edge("identify_entities", "investigate_case")
builder.add_edge("investigate_case", "reason_about_case")
builder.add_edge("reason_about_case", END)

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [65]:
config = {
    "configurable": {
        "thread_id": "customer_C10230_reasoning_test_2"
    }
}

result = graph.invoke(
    {
        "query": "Customer C10230 says my payment was deducted but my order was cancelled. Ticket T03298"
    },
    config
)

print(result.keys())

dict_keys(['query', 'customer_id', 'order_id', 'ticket_id', 'current_case', 'customer_history', 'similar_cases', 'transaction_data', 'case_reasoning'])


In [66]:
print("\n========== CASE REASONING ==========\n")

print("Issue Summary:")
print(result["case_reasoning"]["issue_summary"])

print("\nLikely Root Cause:")
print(result["case_reasoning"]["likely_root_cause"])

print("\nEvidence:")
for evidence in result["case_reasoning"]["evidence"]:
    print("-", evidence)

print("\nConfidence:")
print(result["case_reasoning"]["confidence"])

print("\nRecommended Action:")
print(result["case_reasoning"]["recommended_action"])


========== CASE REASONING ==========

Issue Summary:
Customer reports that payment was deducted but the order was cancelled without approval.

Likely Root Cause:
Payment-order reconciliation failure leading to mismatched payment and order status.

Evidence:
- Current case fields: order_status = 'CANCELLED', payment_status = 'FAILED', refund_status = 'COMPLETED', root_cause = 'Payment-order reconciliation failure' with confidence 0.97.
- Resolution text for current case explicitly states payment-order reconciliation failure identified and corrective action applied.
- Similar organizational cases (T08410, T02733, T09406, T02330, T07109) all attribute order cancellation after payment to payment-order reconciliation failure.
- Transaction data shows no alternative failure indicators (e.g., risk engine false positive, gateway latency).

Confidence:
High

Recommended Action:
Confirm with the customer that the refund of 15,358.12 has been successfully processed, provide a summary of the corr

In [67]:
# --------------------------------
# Policy Validation Node
# --------------------------------

def validate_policy(state: SupportState):

    reasoning = state.get("case_reasoning", {})
    current_case = state.get("current_case", {})

    recommended_action = reasoning.get(
        "recommended_action",
        ""
    )

    refund_requested = current_case.get(
        "refund_requested",
        False
    )

    refund_status = current_case.get(
        "refund_status",
        "NOT_REQUESTED"
    )

    resolution_status = current_case.get(
        "resolution_status",
        ""
    )

    confidence = reasoning.get(
        "confidence",
        "Low"
    )


    # --------------------------------
    # Policy Rules
    # --------------------------------

    policy_allowed = True
    policy_reason = ""


    # Rule 1: Low-confidence cases
    if confidence == "Low":

        policy_allowed = False

        policy_reason = (
            "Low reasoning confidence. "
            "Human review is required before taking an action."
        )


    # Rule 2: Refund already completed
    elif refund_status == "COMPLETED":

        policy_allowed = False

        policy_reason = (
            "Refund has already been completed. "
            "A second refund should not be created."
        )


    # Rule 3: Refund requested but case is not resolved
    elif refund_requested and resolution_status != "Resolved":

        policy_allowed = True

        policy_reason = (
            "Refund is requested and the case is not yet resolved. "
            "Refund eligibility can proceed to action verification."
        )


    # Rule 4: Normal case
    else:

        policy_allowed = True

        policy_reason = (
            "The recommended action is permitted based on "
            "the currently available case information."
        )


    return {
        "policy_allowed": policy_allowed,
        "policy_reason": policy_reason,
        "validated_action": recommended_action
    }

In [68]:
class SupportState(TypedDict, total=False):

    query: str

    customer_id: Optional[str]
    order_id: Optional[str]
    ticket_id: Optional[str]

    current_case: Dict[str, Any]
    customer_history: List[Dict[str, Any]]
    similar_cases: List[Dict[str, Any]]
    transaction_data: Dict[str, Any]

    case_reasoning: Dict[str, Any]

    policy_allowed: bool
    policy_reason: str
    validated_action: str

    final_response: str

In [70]:
policy_result = validate_policy(result)

print("POLICY ALLOWED:", policy_result["policy_allowed"])

print("\nPOLICY REASON:")
print(policy_result["policy_reason"])

print("\nVALIDATED ACTION:")
print(policy_result["validated_action"])

POLICY ALLOWED: False

POLICY REASON:
Refund has already been completed. A second refund should not be created.

VALIDATED ACTION:
Confirm with the customer that the refund of 15,358.12 has been successfully processed, provide a summary of the corrective actions taken on the reconciliation system, and monitor for any recurrence. Log the case as resolved and schedule a follow‑up check on the reconciliation pipeline to prevent future incidents.


In [71]:
# --------------------------------
# Refund Action Tool
# --------------------------------

def create_refund(state: SupportState):

    current_case = state.get("current_case", {})

    order_id = current_case.get("order_id")
    payment_id = current_case.get("payment_id")
    refund_status = current_case.get("refund_status")
    refund_amount = current_case.get("order_amount", 0)

    # --------------------------------
    # Safety Check
    # --------------------------------

    if refund_status == "COMPLETED":

        return {
            "action_status": "BLOCKED",
            "action_type": "CREATE_REFUND",
            "action_message": (
                f"Refund already completed for order {order_id}. "
                "No duplicate refund was created."
            )
        }

    if not order_id or not payment_id:

        return {
            "action_status": "FAILED",
            "action_type": "CREATE_REFUND",
            "action_message": (
                "Missing order or payment information. "
                "Refund cannot be created."
            )
        }

    # --------------------------------
    # Simulated Refund
    # --------------------------------

    return {
        "action_status": "COMPLETED",
        "action_type": "CREATE_REFUND",
        "action_message": (
            f"Simulated refund of ₹{refund_amount} "
            f"created for order {order_id}."
        )
    }

In [72]:
action_result = create_refund(result)

print("ACTION STATUS:")
print(action_result["action_status"])

print("\nACTION TYPE:")
print(action_result["action_type"])

print("\nACTION MESSAGE:")
print(action_result["action_message"])

ACTION STATUS:
BLOCKED

ACTION TYPE:
CREATE_REFUND

ACTION MESSAGE:
Refund already completed for order ORD-106754. No duplicate refund was created.


In [73]:
action_result = create_refund(result)

print("ACTION STATUS:")
print(action_result["action_status"])

print("\nACTION TYPE:")
print(action_result["action_type"])

print("\nACTION MESSAGE:")
print(action_result["action_message"])

ACTION STATUS:
BLOCKED

ACTION TYPE:
CREATE_REFUND

ACTION MESSAGE:
Refund already completed for order ORD-106754. No duplicate refund was created.


In [74]:
def execute_action(state: SupportState):

    # If policy does not allow the action
    if not state.get("policy_allowed", False):
        return {
            "action_status": "BLOCKED",
            "action_type": "POLICY_BLOCKED",
            "action_message": state.get(
                "policy_reason",
                "Action blocked by policy."
            )
        }

    # Execute the refund tool
    action_result = create_refund(state)

    return {
        "action_status": action_result["action_status"],
        "action_type": action_result["action_type"],
        "action_message": action_result["action_message"]
    }

In [75]:
builder = StateGraph(SupportState)

builder.add_node("identify_entities", identify_entities)
builder.add_node("investigate_case", investigate_case)
builder.add_node("reason_about_case", reason_about_case)
builder.add_node("validate_policy", validate_policy)
builder.add_node("execute_action", execute_action)

builder.add_edge(START, "identify_entities")
builder.add_edge("identify_entities", "investigate_case")
builder.add_edge("investigate_case", "reason_about_case")
builder.add_edge("reason_about_case", "validate_policy")
builder.add_edge("validate_policy", "execute_action")
builder.add_edge("execute_action", END)

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [76]:
config = {
    "configurable": {
        "thread_id": "customer_C10230_action_test"
    }
}

result = graph.invoke(
    {
        "query": "Customer C10230 says my payment was deducted but my order was cancelled. Ticket T03298"
    },
    config
)

In [77]:
print("POLICY:")
print(result["policy_allowed"])
print(result["policy_reason"])

print("\nACTION STATUS:")
print(result["action_status"])

print("\nACTION TYPE:")
print(result["action_type"])

print("\nACTION MESSAGE:")
print(result["action_message"])

POLICY:
False
Refund has already been completed. A second refund should not be created.

ACTION STATUS:


KeyError: 'action_status'

In [78]:
from typing import TypedDict, Optional, List, Dict, Any

class SupportState(TypedDict, total=False):

    query: str

    customer_id: Optional[str]
    order_id: Optional[str]
    ticket_id: Optional[str]

    current_case: Dict[str, Any]
    customer_history: List[Dict[str, Any]]
    similar_cases: List[Dict[str, Any]]
    transaction_data: Dict[str, Any]

    case_reasoning: Dict[str, Any]

    policy_allowed: bool
    policy_reason: str
    validated_action: str

    action_status: str
    action_type: str
    action_message: str

    final_response: str

In [79]:
def execute_action(state: SupportState):

    if not state.get("policy_allowed", False):

        return {
            "action_status": "BLOCKED",
            "action_type": "POLICY_BLOCKED",
            "action_message": state.get(
                "policy_reason",
                "Action blocked by policy."
            )
        }

    action_result = create_refund(state)

    return {
        "action_status": action_result["action_status"],
        "action_type": action_result["action_type"],
        "action_message": action_result["action_message"]
    }

In [80]:
builder = StateGraph(SupportState)

builder.add_node("identify_entities", identify_entities)
builder.add_node("investigate_case", investigate_case)
builder.add_node("reason_about_case", reason_about_case)
builder.add_node("validate_policy", validate_policy)
builder.add_node("execute_action", execute_action)

builder.add_edge(START, "identify_entities")
builder.add_edge("identify_entities", "investigate_case")
builder.add_edge("investigate_case", "reason_about_case")
builder.add_edge("reason_about_case", "validate_policy")
builder.add_edge("validate_policy", "execute_action")
builder.add_edge("execute_action", END)

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [81]:
config = {
    "configurable": {
        "thread_id": "C10230_action_test_2"
    }
}

result = graph.invoke(
    {
        "query": "Customer C10230 says my payment was deducted but my order was cancelled. Ticket T03298"
    },
    config
)

In [82]:
print("POLICY:")
print(result["policy_allowed"])
print(result["policy_reason"])

print("\nACTION STATUS:")
print(result["action_status"])

print("\nACTION TYPE:")
print(result["action_type"])

print("\nACTION MESSAGE:")
print(result["action_message"])

POLICY:
False
Refund has already been completed. A second refund should not be created.

ACTION STATUS:
BLOCKED

ACTION TYPE:
POLICY_BLOCKED

ACTION MESSAGE:
Refund has already been completed. A second refund should not be created.


In [83]:
def verify_action(state: SupportState):

    action_status = state.get("action_status")
    action_type = state.get("action_type")
    action_message = state.get("action_message")

    if action_status == "COMPLETED":
        verification_status = "VERIFIED"
        verification_message = (
            f"Action {action_type} completed successfully. "
            "The system confirmed the action result."
        )

    elif action_status == "BLOCKED":
        verification_status = "NOT_REQUIRED"
        verification_message = (
            f"Action was blocked before execution. "
            f"Reason: {action_message}"
        )

    else:
        verification_status = "FAILED"
        verification_message = (
            f"Action {action_type} was not completed successfully. "
            f"Result: {action_message}"
        )

    return {
        "verification_status": verification_status,
        "verification_message": verification_message
    }

In [84]:
class SupportState(TypedDict, total=False):

    query: str

    customer_id: Optional[str]
    order_id: Optional[str]
    ticket_id: Optional[str]

    current_case: Dict[str, Any]
    customer_history: List[Dict[str, Any]]
    similar_cases: List[Dict[str, Any]]
    transaction_data: Dict[str, Any]

    case_reasoning: Dict[str, Any]

    policy_allowed: bool
    policy_reason: str
    validated_action: str

    action_status: str
    action_type: str
    action_message: str

    verification_status: str
    verification_message: str

    final_response: str

In [85]:
builder = StateGraph(SupportState)

builder.add_node("identify_entities", identify_entities)
builder.add_node("investigate_case", investigate_case)
builder.add_node("reason_about_case", reason_about_case)
builder.add_node("validate_policy", validate_policy)
builder.add_node("execute_action", execute_action)
builder.add_node("verify_action", verify_action)

builder.add_edge(START, "identify_entities")
builder.add_edge("identify_entities", "investigate_case")
builder.add_edge("investigate_case", "reason_about_case")
builder.add_edge("reason_about_case", "validate_policy")
builder.add_edge("validate_policy", "execute_action")
builder.add_edge("execute_action", "verify_action")
builder.add_edge("verify_action", END)

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [86]:
config = {
    "configurable": {
        "thread_id": "C10230_verification_test"
    }
}

result = graph.invoke(
    {
        "query": "Customer C10230 says my payment was deducted but my order was cancelled. Ticket T03298"
    },
    config
)

In [87]:
print("POLICY:")
print(result["policy_allowed"])
print(result["policy_reason"])

print("\nACTION:")
print(result["action_status"])
print(result["action_type"])
print(result["action_message"])

print("\nVERIFICATION:")
print(result["verification_status"])
print(result["verification_message"])

POLICY:
False
Refund has already been completed. A second refund should not be created.

ACTION:
BLOCKED
POLICY_BLOCKED
Refund has already been completed. A second refund should not be created.

VERIFICATION:
NOT_REQUIRED
Action was blocked before execution. Reason: Refund has already been completed. A second refund should not be created.


In [88]:
def collect_customer_feedback(state: SupportState):

    feedback = state.get("customer_feedback")

    if not feedback:
        return {
            "feedback_status": "WAITING",
        }

    feedback = feedback.lower().strip()

    satisfied_words = [
        "yes",
        "satisfied",
        "resolved",
        "fixed",
        "thank you",
        "works"
    ]

    not_satisfied_words = [
        "no",
        "not resolved",
        "not fixed",
        "still",
        "issue remains",
        "not satisfied"
    ]

    if any(word in feedback for word in not_satisfied_words):
        return {
            "feedback_status": "NOT_SATISFIED"
        }

    if any(word in feedback for word in satisfied_words):
        return {
            "feedback_status": "SATISFIED"
        }

    return {
        "feedback_status": "UNCLEAR"
    }

In [89]:
test_state = {
    **result,
    "customer_feedback": "No, my issue is still not resolved"
}

feedback_result = collect_customer_feedback(test_state)

print(feedback_result)

{'feedback_status': 'NOT_SATISFIED'}


In [90]:
test_state = {
    **result,
    "customer_feedback": "Yes, my issue is resolved. Thank you."
}

feedback_result = collect_customer_feedback(test_state)

print(feedback_result)

{'feedback_status': 'SATISFIED'}


In [91]:
builder = StateGraph(SupportState)

builder.add_node("identify_entities", identify_entities)
builder.add_node("investigate_case", investigate_case)
builder.add_node("reason_about_case", reason_about_case)
builder.add_node("validate_policy", validate_policy)
builder.add_node("execute_action", execute_action)
builder.add_node("verify_action", verify_action)
builder.add_node("collect_customer_feedback", collect_customer_feedback)

builder.add_edge(START, "identify_entities")
builder.add_edge("identify_entities", "investigate_case")
builder.add_edge("investigate_case", "reason_about_case")
builder.add_edge("reason_about_case", "validate_policy")
builder.add_edge("validate_policy", "execute_action")
builder.add_edge("execute_action", "verify_action")
builder.add_edge("verify_action", "collect_customer_feedback")

builder.add_edge("collect_customer_feedback", END)

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

In [92]:
config = {
    "configurable": {
        "thread_id": "C10230_feedback_test"
    }
}

result = graph.invoke(
    {
        "query": "Customer C10230 says my payment was deducted but my order was cancelled. Ticket T03298",
        "customer_feedback": "No, my issue is still not resolved"
    },
    config
)

In [93]:
print("ACTION:")
print(result["action_status"])
print(result["action_message"])

print("\nVERIFICATION:")
print(result["verification_status"])
print(result["verification_message"])

print("\nCUSTOMER FEEDBACK:")
print(result["customer_feedback"])

print("\nFEEDBACK STATUS:")
print(result["feedback_status"])

ACTION:
BLOCKED
Refund has already been completed. A second refund should not be created.

VERIFICATION:
NOT_REQUIRED
Action was blocked before execution. Reason: Refund has already been completed. A second refund should not be created.

CUSTOMER FEEDBACK:


KeyError: 'customer_feedback'